# Phase 2 实验运行器

这个 notebook 用于在 Google Colab 上运行 Phase 2 的损失函数组合实验。

## 实验概述

- **Phase 2.1**: 13 个损失函数 × 3 个种子 × 1 个权重上限 = 39 次运行
- **Phase 2.2**: 3 个最佳损失函数 × 6 个种子 × 2 个权重上限 = 36 次运行

## 四种损失函数变体

1. **Variant 1**: IMADL + M2 线性组合 (7 个)
2. **Variant 2**: IMADL + GMADL 加权组合 (3 个)
3. **Variant 3**: M2 鲁棒性增强 (3 个)
4. **Variant 4**: 自适应混合 (3 个)

## Step 1: 挂载 Google Drive

In [ ]:
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

# 验证挂载
DRIVE_ROOT = "/content/drive/MyDrive/FYP"
assert os.path.exists(DRIVE_ROOT), "Drive not mounted!"
print(f"✓ Drive mounted at {DRIVE_ROOT}")

# 显示目录结构
!ls -lh {DRIVE_ROOT}

## Step 2: 克隆/更新代码仓库

In [ ]:
import os

os.chdir("/content")

# 克隆仓库（首次运行）
if not os.path.exists("FYP"):
    print("Cloning repository...")
    !git clone https://github.com/YOUR_USERNAME/FYP.git
    os.chdir("FYP")
    !git checkout phase2/loss-combinations
else:
    # 更新代码（后续运行）
    print("Updating repository...")
    os.chdir("FYP")
    !git fetch origin
    !git checkout phase2/loss-combinations
    !git pull origin phase2/loss-combinations

print("\n✓ Code ready")
!pwd
!git branch

## Step 3: 安装依赖

In [ ]:
# 安装 Python 依赖
!pip install torch pandas numpy matplotlib seaborn -q

print("✓ Dependencies installed")

# 验证安装
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Step 4: 验证数据文件

In [ ]:
# 检查数据文件
DATA_DIR = f"{DRIVE_ROOT}/data"

print(f"Checking data directory: {DATA_DIR}")
!ls -lh {DATA_DIR}/*.csv

# 统计数据文件
import glob
csv_files = glob.glob(f"{DATA_DIR}/*.csv")
print(f"\n✓ Found {len(csv_files)} CSV files")

## Step 5: 运行 Phase 2.1 实验（39 runs）

这将运行所有 13 个损失函数，每个使用 3 个随机种子，权重上限为 5%。

In [ ]:
# Phase 2.1 配置
TRAIN_START = "1990-01"
TRAIN_END = "1994-12"
TEST_START = "1995-01"
TEST_MONTHS = 24
MAX_EPOCHS = 20
BATCH_SIZE = 1024

print("Starting Phase 2.1 experiments...")
print(f"Train: {TRAIN_START} to {TRAIN_END}")
print(f"Test: {TEST_START} for {TEST_MONTHS} months")
print(f"Expected runs: 13 losses × 3 seeds = 39 runs")
print("-" * 80)

# 运行实验
!python run_phase2_robustness.py \
  --drive-root {DRIVE_ROOT} \
  --data-dir {DATA_DIR} \
  --train-start {TRAIN_START} \
  --train-end {TRAIN_END} \
  --test-start {TEST_START} \
  --test-months {TEST_MONTHS} \
  --max-epochs {MAX_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --matrix-mode light \
  --resume-mode auto \
  --skip-existing

## Step 6: 聚合 Phase 2.1 结果

In [ ]:
print("Aggregating Phase 2.1 results...")

!python aggregate_phase2_results.py \
  --drive-root {DRIVE_ROOT} \
  --seeds 42,52,62 \
  --caps 0.05

print("\n✓ Aggregation complete")

## Step 7: 分析 Phase 2.1 结果并选择 Top 3

In [ ]:
import pandas as pd

# 读取聚合结果
summary_file = f"{DRIVE_ROOT}/phase2/phase2_grouped_summary.csv"
summary = pd.read_csv(summary_file)

print("Phase 2.1 Results Summary")
print("=" * 80)

# 按 Sharpe 排序
summary_sorted = summary.sort_values("sharpe_ratio_mean", ascending=False)

print("\nTop 10 Loss Functions by Mean Sharpe Ratio:")
print("-" * 80)
top10 = summary_sorted.head(10)
for idx, row in top10.iterrows():
    print(f"{row['loss']:25s} | Sharpe: {row['sharpe_ratio_mean']:6.3f} ± {row['sharpe_ratio_std']:5.3f} | CV: {row['sharpe_cv']:5.3f} | Fail: {row['failure_rate']:4.1f}%")

# 选择 Top 3
top3 = summary_sorted.head(3)
top3_losses = top3["loss"].tolist()

print("\n" + "=" * 80)
print("Selected Top 3 for Phase 2.2:")
print("=" * 80)
for i, loss in enumerate(top3_losses, 1):
    row = top3[top3["loss"] == loss].iloc[0]
    print(f"{i}. {loss}")
    print(f"   Sharpe: {row['sharpe_ratio_mean']:.4f} ± {row['sharpe_ratio_std']:.4f}")
    print(f"   CV: {row['sharpe_cv']:.4f}")
    print(f"   Failure Rate: {row['failure_rate']:.1f}%")
    print()

# 保存 Top 3 列表
top3_file = f"{DRIVE_ROOT}/phase2/top3_losses.txt"
with open(top3_file, "w") as f:
    f.write("\n".join(top3_losses))

print(f"✓ Top 3 saved to {top3_file}")

## Step 8: 可视化 Phase 2.1 结果

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 设置样式
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# 1. Sharpe Ratio 对比
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Sharpe Ratio
ax1 = axes[0, 0]
summary_sorted.plot(x="loss", y="sharpe_ratio_mean", kind="bar", ax=ax1, color="steelblue")
ax1.axhline(y=0.464, color='green', linestyle='--', label='IMADL Baseline')
ax1.axhline(y=0.914, color='red', linestyle='--', label='M2 Baseline')
ax1.set_title("Mean Sharpe Ratio by Loss Function", fontsize=14, fontweight='bold')
ax1.set_xlabel("Loss Function")
ax1.set_ylabel("Mean Sharpe Ratio")
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Coefficient of Variation
ax2 = axes[0, 1]
summary_sorted.plot(x="loss", y="sharpe_cv", kind="bar", ax=ax2, color="coral")
ax2.axhline(y=0.892, color='green', linestyle='--', label='IMADL CV')
ax2.axhline(y=1.396, color='red', linestyle='--', label='M2 CV')
ax2.set_title("Coefficient of Variation (Stability)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Loss Function")
ax2.set_ylabel("CV (lower = more stable)")
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

# Failure Rate
ax3 = axes[1, 0]
summary_sorted.plot(x="loss", y="failure_rate", kind="bar", ax=ax3, color="salmon")
ax3.axhline(y=0, color='green', linestyle='--', label='IMADL (0%)')
ax3.axhline(y=33, color='red', linestyle='--', label='M2 (33%)')
ax3.set_title("Failure Rate (Sharpe < 0)", fontsize=14, fontweight='bold')
ax3.set_xlabel("Loss Function")
ax3.set_ylabel("Failure Rate (%)")
ax3.legend()
ax3.tick_params(axis='x', rotation=45)

# Cumulative Return
ax4 = axes[1, 1]
summary_sorted.plot(x="loss", y="cumulative_return_mean", kind="bar", ax=ax4, color="mediumseagreen")
ax4.set_title("Mean Cumulative Return", fontsize=14, fontweight='bold')
ax4.set_xlabel("Loss Function")
ax4.set_ylabel("Cumulative Return (%)")
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/phase2/phase21_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Visualization saved to {DRIVE_ROOT}/phase2/phase21_visualization.png")

## Step 9: 运行 Phase 2.2 实验（可选）

仅对 Top 3 损失函数运行扩展验证：
- 6 个随机种子（42, 52, 62, 72, 82, 92）
- 2 个权重上限（0.05 和 无限制）
- 48 个月测试期

总计：3 losses × 6 seeds × 2 caps = 36 runs

In [ ]:
# 读取 Top 3
with open(f"{DRIVE_ROOT}/phase2/top3_losses.txt") as f:
    top3_losses = f.read().strip().split("\n")

print(f"Running Phase 2.2 for: {', '.join(top3_losses)}")
print(f"Expected runs: 3 losses × 6 seeds × 2 caps = 36 runs")
print("-" * 80)

# Phase 2.2 配置
TEST_MONTHS_P22 = 48  # 扩展到 48 个月
SEEDS_P22 = "42,52,62,72,82,92"

!python run_phase2_robustness.py \
  --drive-root {DRIVE_ROOT} \
  --data-dir {DATA_DIR} \
  --losses {",".join(top3_losses)} \
  --seeds {SEEDS_P22} \
  --train-start {TRAIN_START} \
  --train-end {TRAIN_END} \
  --test-start {TEST_START} \
  --test-months {TEST_MONTHS_P22} \
  --max-epochs {MAX_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --matrix-mode full \
  --resume-mode auto \
  --skip-existing

## Step 10: 聚合 Phase 2.2 结果

In [ ]:
print("Aggregating Phase 2.2 results...")

!python aggregate_phase2_results.py \
  --drive-root {DRIVE_ROOT} \
  --losses {",".join(top3_losses)} \
  --seeds {SEEDS_P22} \
  --caps 0.05,None \
  --output-dir {DRIVE_ROOT}/phase2/phase22

print("\n✓ Phase 2.2 aggregation complete")

## Step 11: 最终报告

In [ ]:
# 显示最终报告
report_file = f"{DRIVE_ROOT}/phase2/phase2_summary_report.txt"
if os.path.exists(report_file):
    with open(report_file, "r") as f:
        print(f.read())
else:
    print("Report not found. Run aggregation first.")

## 完成！

所有结果已保存到 Google Drive:
- 原始结果: `{DRIVE_ROOT}/phase2/results/`
- Checkpoints: `{DRIVE_ROOT}/phase2/checkpoints/`
- 日志: `{DRIVE_ROOT}/phase2/logs/`
- 聚合结果: `{DRIVE_ROOT}/phase2/phase2_raw_runs.csv`
- 汇总统计: `{DRIVE_ROOT}/phase2/phase2_grouped_summary.csv`
- 报告: `{DRIVE_ROOT}/phase2/phase2_summary_report.txt`